In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

# SMART-DS grid geometry: all of this location's 2016 base-timeseries feeders.
def find_location_root(location_name="sfo"):
    for base in (Path.cwd(), *Path.cwd().parents):
        if base.name == location_name and (base / "01_grid").exists():
            return base
        candidate = base / "locations" / location_name
        if (candidate / "01_grid" / "sds_plot.ipynb").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate locations/{location_name} from {Path.cwd()}")

def smartds_subregion(path):
    parts = path.parts
    if "scenarios" in parts:
        return parts[parts.index("scenarios") - 1]
    if "base_timeseries" in parts:
        return parts[parts.index("base_timeseries") - 1]
    raise ValueError(f"Could not infer SMART-DS subregion from {path}")

location_root = find_location_root()
location_name = location_root.name
root = location_root / "data" / "smart_ds" / "2016"
patterns = (
    "*/scenarios/base_timeseries/geojson/*.json",
    "*/base_timeseries/geojson/*.json",
)
paths = sorted({path for pattern in patterns for path in root.glob(pattern)})
if not paths:
    joined_patterns = ", ".join(patterns)
    raise FileNotFoundError(f"No SMART-DS GeoJSON feeders found under {root} with patterns: {joined_patterns}")

grid = pd.concat(
    [gpd.read_file(path).assign(subregion=smartds_subregion(path), feeder=path.stem) for path in paths],
    ignore_index=True,
)
ax = grid[grid["type"].eq("Line")].plot(figsize=(9, 9), linewidth=0.25, color="tab:blue", alpha=0.55)
grid[~grid["type"].eq("Line")].plot(ax=ax, markersize=0.15, color="black", alpha=0.25)
ax.set_title(f"SMART-DS {location_name.title()} base grid ({len(paths)} feeders)")
ax.set_axis_off()

In [ ]:
import math
import matplotlib.pyplot as plt

def plot_smartds_subregions(grid, *, ncols=3, line_width=0.25, node_size=0.15):
    subregions = sorted(grid["subregion"].dropna().unique())
    nrows = math.ceil(len(subregions) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4.5 * nrows), constrained_layout=True)
    axes = list(axes.ravel())

    for ax, subregion in zip(axes, subregions):
        subset = grid[grid["subregion"].eq(subregion)]
        lines = subset[subset["type"].eq("Line")]
        points = subset[~subset["type"].eq("Line")]
        lines.plot(ax=ax, linewidth=line_width, color="tab:blue", alpha=0.55)
        points.plot(ax=ax, markersize=node_size, color="black", alpha=0.25)
        ax.set_title(f"{subregion} ({subset['feeder'].nunique()} feeders)")
        ax.set_axis_off()

    for ax in axes[len(subregions):]:
        ax.set_axis_off()

    fig.suptitle(f"SMART-DS {location_name.title()} base grid by subregion", y=1.02)
    return fig, axes[:len(subregions)]

fig, axes = plot_smartds_subregions(grid)